In [1]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

In [2]:
def extract_text_from_pdf(pdf_path):
    text=""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text=page.extract_text()
                if page_text:
                    text+=page_text
        if text.strip():
            return text.strip()
        
    except Exception as e:
        print("Direct text extraction failed ", e)


    print("Falling back to OCR for image based pdf")
    try:
        images=convert_from_path(pdf_path)
        for image in images:
            page_text=pytesseract.image_to_string(image)
            text+=page_text+"\n"
    
    except Exception as e:
        print("OCR failed ", e)

pdf_path="resume.pdf"
extracted_text=extract_text_from_pdf(pdf_path)
        

In [3]:
extracted_text

In [9]:
import dotenv
import os
import google.generativeai as genai

dotenv.load_dotenv()
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

genai.configure(api_key="API KEY
model=genai.GenerativeModel("gemini-1.5-flash")


In [10]:
response=model.generate_content("What is the capital of India?")

In [11]:
response.text

'The capital of India is **New Delhi**.\n'

In [25]:
from fpdf import FPDF

def analyze_resume(resume_text,job_description=None):
    if not resume_text:
        return {"error":"Resume text is required for analysis"}
    
    base_prompt= f"""
        You are an experienced HR with experience in the field of any job role. 
        Your task is to review the provided resume. 
        Please share your your professional evaluation as well as a score on whether the candidate's profile aligns with the role.
        Also mention skills he already has and suggest some skills to improve his resume. 
        Also suggest some course he might take to imrove his skills.
        Hightlight the strengths and weaknesses.

        Resume:
        {resume_text}

    """
    
    if job_description:
        base_prompt+= f"""
            Additionally compare this resume to following job description.

            job description:
            {job_description}

            Highlight the strengths and weaknesses of the applicant in relation to specified job description.
        """

    response=model.generate_content(base_prompt)
    analysis=response.text.strip()

    pdf=FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    for line in analysis.split("\n"):
        pdf.multi_cell(0,10,line)

    try:
        pdf.output("gemini_response.pdf")
        print("PDF created successfully.")
        
    except Exception as e:
        print("Error saving PDF:", e)


In [30]:
job_des="""
An Embedded Engineer designs, develops, and tests software and hardware systems for specialized devices, ensuring seamless integration of firmware and optimizing system performance. They work on low-level programming, real-time operating systems, and hardware-software integration, requiring strong skills in C/C++, debugging, and collaboration with cross-functional teams. 
Key Responsibilities:
Software Development: Designing and implementing embedded software solutions for microcontrollers and microprocessors. 
Hardware-Software Integration: Collaborating with hardware engineers to define requirements and ensure seamless integration. 
Coding: Writing efficient, modular, and well-documented code, often in C/C++. 
Testing & Debugging: Conducting unit testing, debugging, and troubleshooting software to identify and resolve issues. 
Performance Optimization: Analyzing and enhancing efficiency, stability, and scalability of system resources. 
Collaboration: Working with testing teams, hardware engineers, and other stakeholders. 
Staying Updated: Keeping abreast of advancements in embedded software development, including new tools and technologies. 
Essential Skills:
Programming Languages: Proficiency in C and C++, with experience in assembly language for microcontrollers. 
Microcontrollers & Microprocessors: Familiarity with various architectures and their operation. 
Real-Time Operating Systems (RTOS): Experience with real-time operating systems and embedded Linux environments. 
Hardware Understanding: Knowledge of hardware architecture, debugging, and troubleshooting. 
Software Testing: Experience with testing methodologies and tools for embedded systems. 
Communication: Excellent communication and collaboration skills for working with cross-functional teams. 
Problem-Solving: Strong analytical and problem-solving skills to debug complex issues. """

In [31]:
analyze_resume(extracted_text,job_des)

PDF created successfully.
